# Simple thresholding-based segmentation of nuclei

## Input

This recipe expects an input folder containing 3D & multichannel ```.nd2``` files.

**NOTE:** single channel images are untested at the moment.

## Output

In the specified output folder (by default a subfolder of the input folder), for each ```.nd2``` file in the input, a TIFF file named ```{input file name (without ending)}_segmented.tif``` will be created. If instance segmentation is requested, it will be done via standard Watershed on the EDT of the mask.

Furthermore, a visualization of segmentations in z-maximum-projections will be created in a subfolder.

## 0) Imports and Function definitions

Run this once

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
from skimage.io import imsave
from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, ball, remove_small_objects, remove_small_holes
from skimage.color import label2rgb
from skimage.exposure import rescale_intensity
from scipy.ndimage import gaussian_filter
from nd2 import ND2File
from skimage.morphology.extrema import h_maxima
from skimage.measure import label
from skimage.segmentation import watershed
from edt import edt

def load_single_channel_from_nd2(file_path, channel=0):
    with ND2File(file_path) as reader:

        if isinstance(channel, str):
            # nice OC name without whitespace
            channel_names = list(map(lambda s: s.channel.name.strip().replace(' ', '-'), reader.metadata.channels))
            
            # try to find specified channel, otherwise error and list available channels
            try:
                channel_idx = channel_names.index(channel)
            except ValueError:
                raise ValueError(f'channel {channel} not found in file. available channels: {channel_names}')
                
        else:
            channel_idx = channel

        img = np.array(reader.to_dask()[:,channel_idx])
        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]
    
    return img, pixel_size

def threshold_segmentation(img, blur_sigma, small_hole_size, small_hole_size_perplane, small_object_size, closing_radius):
    
    # blur slightly and threshold via Otsu
    segmented = img > threshold_otsu(gaussian_filter(img, blur_sigma))

    # some morphological cleanup
    if closing_radius > 0:
        segmented = binary_closing(segmented, ball(closing_radius))

    # remove small holes per plane first
    for plane in range(segmented.shape[0]):
        segmented[plane] = remove_small_holes(segmented[plane], small_hole_size_perplane)

    segmented = remove_small_holes(segmented, small_hole_size)
    segmented = remove_small_objects(segmented, small_object_size)

    return segmented

def get_segmentation_visualization(labels, img):
    # project and rescale to 8bit range
    img_projected = rescale_intensity(img.max(axis=0), in_range=tuple(np.quantile(img, (0.02, 0.9995))), out_range='uint8')
    # overlay labels
    # NOTE: bg_label is 0 in our binary segmentation
    visualization_projection = label2rgb(labels.max(axis=0), img_projected, bg_label=0)
    # float result to 8bit uint
    visualization_projection = (visualization_projection * 255).astype(np.uint8)
    return visualization_projection

def imsave_nowarnings(file, img, **kwargs):
    # catch low contrast warning
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        imsave(file, img, **kwargs)

def edt_watershed_instance_segmentation(mask, h_maxima_threshold, pixel_size):
    dt = edt(mask, anisotropy=pixel_size)
    maxima = h_maxima(dt, h_maxima_threshold)
    segmented_instance = watershed(-dt, label(maxima), mask=mask, connectivity=2)
    return(segmented_instance)



## 1) Set input and parameters

In [ ]:
# path containing files to visualize
in_path = Path('/data/agl_data/NanoFISH/Gabi/GS075_20230818_K562-EVI1-GFP_t(3-8)_EVI-CTRL/')

# path to which output is saved
# default: put results in subdirectory called 'segmentation-threshold'
out_path = in_path / 'segmentation-threshold'

## parameters for segmentation
# sigma of blur to apply before thresholding
blur_sigma = 1
# size (in pixels) of small objects/small holes to discard
small_hole_size = 50000
small_hole_size_perplane = 5000
small_object_size = 50000
# radius of binary closing applied to mask (can be slow, set to 0 to skip)
closing_radius = 2

# channel to use for segmentation. may be either interger index (e.g., 0) or the name of the channel (OC in NIS)
channel_for_segmentation = '405-CSU-W1'

# do instance segmentation?
# can be: None/False - don't do instance segmentation
# 'connected-components' - only do connected components labelling
# 'watershed' - do watershed transform on edt of mask
instance_segmentation = 'connected-components'
# if instance segmentation by watershed, what is the required prominence of EDT maxima to be considered as seed points
# lower values: oversegmentation, higher values: undersegmentation
h_maxima_threshold = 1.5

# whether to save a simple png visualization of segmentation results in a subfolder
save_visualization = True

# how many images to process in parallel
# NOTE: too high didn't seem to help that much, might be better on newer conda env
n_jobs_parallel = 4


parameter_log = {
    'in_path': str(in_path),
    'channel_for_segmentation': channel_for_segmentation,
    'blur_sigma': blur_sigma,
    'small_hole_size': small_hole_size,
    'small_hole_size_perplane': small_hole_size_perplane,
    'small_object_size': small_object_size,
    'instance_segmentation': instance_segmentation,
    'h_maxima_threshold': h_maxima_threshold
}

## 2) Check input files

Run this to get the list of files to process and print for verification

In [ ]:
# get all nd2 files in in_path
in_files = sorted(list(Path(in_path).glob('*.nd2')))

# show for verification
in_files

## 3) run segmentation

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# function to process one file
def process_single_file(in_file):

    img, pixel_size = load_single_channel_from_nd2(in_file, channel=channel_for_segmentation)
    segmented = threshold_segmentation(img, blur_sigma, small_hole_size, small_hole_size_perplane, small_object_size, closing_radius)

    if not instance_segmentation:
        pass
    elif instance_segmentation == 'watershed':
        segmented = edt_watershed_instance_segmentation(segmented, h_maxima_threshold, pixel_size)
    elif instance_segmentation == 'connected-components':
        segmented = label(segmented)
    else:
        raise ValueError(f'instance segmentation method "{instance_segmentation}" not available')

    # convert to 8 or 16 bit as necessary
    segmented = segmented.astype(np.uint16) if segmented.max() > 255 else segmented.astype(np.uint8)
    
    if save_visualization:
        visualization_projection = get_segmentation_visualization(segmented, img)
    else:
        visualization_projection = None
    return segmented, visualization_projection

# make output directories if necessary
if not out_path.exists():
    out_path.mkdir(parents=True)

visualization_path = out_path / 'quick_result_visualization'
if save_visualization and not visualization_path.exists():
    visualization_path.mkdir(parents=True)

# segment in parallel
with ThreadPoolExecutor(n_jobs_parallel) as tpe:
    futures = [tpe.submit(process_single_file, in_file) for in_file in in_files]
    results = []
    for f, in_file in zip(futures, in_files):
        results.append(f.result())
        print(f'segmented {str(in_file)}.')

with open(out_path / 'segmentation_parameters.json', 'w') as fd:
    json.dump(parameter_log, fd, indent=1) 

for (segmented, visualization_projection), in_file in zip(results, in_files):
    
    # make filepath for output
    outfile = out_path / (in_file.stem + '_segmented.tif')
    imsave_nowarnings(str(outfile), segmented)

    if save_visualization:

        # make filepath for output
        outfile_visualization = visualization_path / (in_file.stem + '_segmented_projection.png')            
        imsave_nowarnings(str(outfile_visualization), visualization_projection)

    


In [ ]:
# test on one file
from matplotlib import pyplot as plt

idx = 12
segmented, visualization_projection = process_single_file(in_files[idx])

plt.figure(figsize=(12,12))
plt.imshow(visualization_projection)